# V4 — Gaussian blur shared-memory tile + halo

Owner: Thịnh. V4 kế thừa **đúng V3 đã merge**, chỉ thay Gaussian blur. Nền V3 đã tích hợp V2 S4 và V3 S6; V4 giữ nguyên các stage này.

Chọn Colab **GPU**, rồi Run all sau khi source V4 đã được push lên branch `v4`. Notebook không chứa số đo giả định. Chốt hiệu quả V4 chỉ sau parity, 12 case timing và profiler V3/V4.


In [ ]:
from pathlib import Path
import datetime
import json
import os
import shlex
import shutil
import subprocess

REPO_URL = "https://github.com/nganbee/panorama-stitching-project.git"
BRANCH = "v4"
REPO = Path("/content/panorama-stitching-project")

def run(args, check=True):
    print("$", shlex.join([str(a) for a in args]), flush=True)
    result = subprocess.run([str(a) for a in args], text=True)
    if check and result.returncode:
        raise RuntimeError(f"Command failed: {result.returncode}")
    return result.returncode

if not REPO.exists():
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
run(["git", "status", "--short"])
run(["git", "branch", "--show-current"])
run(["git", "log", "-1", "--oneline"])
if not Path("src/stage_implementations/v4_stage1.cu").is_file():
    raise RuntimeError("Thiếu source V4. Push branch v4 rồi cập nhật checkout Colab bằng git pull --ff-only.")
Path("build").mkdir(exist_ok=True)
Path("out").mkdir(exist_ok=True)


## 1. GPU, compiler và build chung

Giữ `--fmad=false`, không dùng fast math. Một binary cung cấp V0/V1/V2/V3/V4; mọi bản dùng cùng input, flags và GPU session. Tile mặc định 16×16. Có thể build lại với tile 32×8 hoặc 32×16 qua `TILE_FLAGS`, chạy lại toàn bộ parity và lưu kết quả vào thư mục riêng.


In [ ]:
run(["nvidia-smi"])
run(["nvcc", "--version"])
run(["g++", "--version"])
TILE_FLAGS = []  # e.g. ["-DPANORAMA_V4_TILE_X=32", "-DPANORAMA_V4_TILE_Y=8"]
FLAGS = ["nvcc", "-O2", "-std=c++17", "--fmad=false",
         "-DPANORAMA_WITH_CUDA", "-DPANORAMA_WITH_V2", "-DPANORAMA_WITH_V3", "-DPANORAMA_WITH_V4",
         "-Isrc/core_headers", "-Isrc/libraries"] + TILE_FLAGS
SOURCES = [
    "src/integration/v0_pipeline.cpp",
    "src/stage_implementations/v0_stage12.cpp",
    "src/stage_implementations/v0_stage34.cpp",
    "src/stage_implementations/v0_stage5.cpp",
    "src/stage_implementations/v0_stage6.cpp",
    "src/stage_implementations/v1_stage12.cu",
    "src/stage_implementations/v1_stage34.cu",
    "src/stage_implementations/v1_stage5.cu",
    "src/stage_implementations/v1_stage6.cu",
    "src/stage_implementations/v2_stage4.cu",
    "src/stage_implementations/v3_stage6.cu",
    "src/stage_implementations/v4_stage1.cu",
    "src/cuda/cuda_pipeline.cu", "src/cuda/v2_pipeline.cu", "src/cuda/v3_pipeline.cu", "src/cuda/v4_pipeline.cu",
    "src/libraries/stb_impl.cpp",
]
run(FLAGS + ["src/integration/main.cpp"] + SOURCES + ["-o", "build/stitch_v4"])
run(["build/stitch_v4", "--list-versions"])
run(["build/stitch_v4", "--dump-params"])


## 2. Parity blur riêng và lan truyền S2–S4

Test V0/V3/V4, ảnh hằng, impulse góc, checkerboard, random, ảnh 1 pixel/nhỏ hơn halo, kích thước không chia hết tile và input lỗi. Yêu cầu V3/V4 blur bit-identical với cùng flags; so V0 theo `atol=2e-4, rtol=2e-6`. Kiểm tra tiếp keypoints, descriptor bits và ordered matches.


In [ ]:
run(FLAGS + [
    "src/tools/v4/test_v4_blur.cpp",
    "src/stage_implementations/v0_stage12.cpp", "src/stage_implementations/v0_stage34.cpp",
    "src/stage_implementations/v1_stage12.cu", "src/stage_implementations/v1_stage34.cu",
    "src/stage_implementations/v2_stage4.cu",
    "src/stage_implementations/v4_stage1.cu",
    "-o", "build/test_v4_blur",
])
run(["build/test_v4_blur"])
run(["build/stitch_v4", "--selftest", "--version", "4"])


## 3. Parity toàn pipeline V4 so với V0/V3

Test identity, dịch chuyển, input lỗi và no-feature; sau đó chạy lại để kiểm tra trạng thái profiler. So keypoints/matches/inliers, sai số chiếu H ≤1e-3 px, kích thước crop, RGB ≤1 mức so với V0 và RGB bit-identical V3/V4. Test timing kiểm tra tổng stage bằng kernel_total. Case V0 thất bại không được đánh dấu panorama parity PASS.


In [ ]:
run(FLAGS + ["src/tools/v4/test_v4_pipeline.cpp"] + SOURCES + ["-o", "build/test_v4_pipeline"])
run(["build/test_v4_pipeline"])


## 4. Compute Sanitizer

Chạy cả memcheck và synccheck cho test blur để phát hiện truy cập halo và lỗi barrier; pipeline test chạy memcheck. Nếu tool không có, ghi rõ NOT RUN thay vì PASS.


In [ ]:
if shutil.which("compute-sanitizer"):
    run(["compute-sanitizer", "--tool", "memcheck", "--error-exitcode", "1", "build/test_v4_blur"])
    run(["compute-sanitizer", "--tool", "synccheck", "--error-exitcode", "1", "build/test_v4_blur"])
    run(["compute-sanitizer", "--tool", "memcheck", "--error-exitcode", "1", "build/test_v4_pipeline"])
else:
    print("NOT RUN: Compute Sanitizer unavailable. Chưa nghiệm thu memory/synchronization safety.")


## 5. Baseline V3: xác nhận tỷ trọng blur trước phép đo V4

CLI tự chạy **1 warm-up + tối thiểu 5 lần đo**, JSON là trung bình. `blur` gồm sinh coefficients + horizontal + vertical cho cả A/B. Đây là timing kernel; H2D/D2H riêng, E2E gồm allocation/launch/sync/copy. Đọc/ghi ảnh nằm ngoài E2E. Cần profiler ở mục 8 để xác nhận blur thực sự đáng tối ưu.


In [ ]:
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RESULTS = Path("out") / f"v4_benchmark_{STAMP}"
BASELINE = Path("out") / f"v3_baseline_{STAMP}.json"
LEFT = "dataset/location1/large_4k/1.jpg"
RIGHT = "dataset/location1/large_4k/2.jpg"
run(["build/stitch_v4", LEFT, RIGHT, "out/v3_baseline.png",
     "--version", "3", "--runs", "5", "--json", BASELINE])
baseline = json.loads(BASELINE.read_text())
t = baseline["timings_ms"]
print(f"V3 blur={t['blur']:.3f} ms; E2E={t['end_to_end']:.3f} ms")
print(f"Blur share: {100*t['blur']/t['kernel_total']:.2f}% of kernel; "
      f"{100*t['blur']/t['end_to_end']:.2f}% of E2E")


## 6. Đủ 12 case: correctness + V0/V3/V4 timing + JSON

Mỗi case chạy parity riêng, rồi V0/V3/V4 có warm-up và ≥5 timed runs. Mọi ca lỗi được lưu log, ghi failed; script vẫn đo các case còn lại. `summary.csv/json` chứa stage bottleneck, blur/E2E speedup và transfer timings. Kết quả tăng tốc dưới 1 được giữ để phân tích. Tránh so số đo lấy từ các GPU session khác nhau.


In [ ]:
benchmark_status = run(["python", "src/tools/v4/benchmark_v4.py",
    "--binary", "build/stitch_v4", "--parity", "build/test_v4_pipeline",
    "--runs", "5", "--output", RESULTS], check=False)
print("Benchmark exit code:", benchmark_status)
(RESULTS / "build_flags.json").write_text(json.dumps(FLAGS, indent=2), encoding="utf-8")


## 7. Bảng trước/sau, bottleneck và panorama


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

summary = pd.read_csv(RESULTS / "summary.csv")
display(summary)
good = summary[summary["status"] == "ok"].copy()
cols = ["case", "v3_blur_ms", "v4_blur_ms", "blur_speedup_v3_v4",
        "v3_end_to_end_ms", "v4_end_to_end_ms", "e2e_speedup_v3_v4",
        "v3_blur_share_e2e_pct", "v3_bottleneck_stage", "v4_bottleneck_stage"]
if len(good):
    display(good[cols].round(4))
    good.set_index("case")[["blur_speedup_v3_v4", "e2e_speedup_v3_v4"]].plot.bar(figsize=(14, 5))
    plt.axhline(1.0, color="black", linestyle="--")
    plt.ylabel("V3 / V4 speedup")
    plt.tight_layout()
    plt.show()
    case = good.iloc[0]["case"]
    display(Image.open(RESULTS / f"{case}_v3.png"))
    display(Image.open(RESULTS / f"{case}_v4.png"))
assert len(summary) == 12 and len(good) == 12, "Chưa PASS đủ 12 case; xem parity/CLI logs."


## 8. Profiler hỗ trợ kết luận

Bật cell dưới nếu Nsight Systems có sẵn. Lưu profile V3/V4 cùng case/GPU/build. Xem hai kernel blur, copy và launch/sync; profiler làm thay đổi thời gian nên dùng JSON benchmark không bật profiler để báo cáo speedup. Nếu blur nhỏ so E2E, trình bày giới hạn lợi ích dù kernel nhanh hơn. Nếu tool thiếu/quyền counter thiếu, ghi NOT RUN và chạy profiler trên máy CUDA khác trước khi chốt kết luận.


In [ ]:
RUN_PROFILER = False
if RUN_PROFILER and shutil.which("nsys"):
    for version in (3, 4):
        prefix = RESULTS / f"nsys_v{version}_4k"
        run(["nsys", "profile", "--trace=cuda", "--output", prefix,
             "build/stitch_v4", LEFT, RIGHT, RESULTS / f"profile_v{version}.png",
             "--version", str(version), "--runs", "5"])
        # Summary includes kernel durations and memory transfers.
        with (RESULTS / f"nsys_v{version}_stats.txt").open("w") as log:
            subprocess.run(["nsys", "stats", str(prefix) + ".nsys-rep"],
                           stdout=log, stderr=subprocess.STDOUT, text=True, check=True)
else:
    print("NOT RUN: bật RUN_PROFILER=True và dùng runtime có nsys để thu bằng chứng profiling.")


## 9. Bàn giao

Lưu output notebook và tải thư mục RESULTS: environment/build flags, 36 JSON timing, logs parity, summary CSV/JSON, panorama V0/V3/V4 và profiler nếu đã chạy. Viết nhận xét riêng theo case/resolution: blur có đáng kể ở V3 không, tốc độ blur thay đổi bao nhiêu, E2E thay đổi bao nhiêu, case chậm hơn và lý do. Không kết luận V4 nhanh hơn trước khi có số đo. Source V3/V4 dùng chung điều phối, chỉ callback blur khác nhau; residency/constant coefficients/fusion chưa nằm trong V4.
